In [ ]:
import random as rd

import dimod
import networkx as nx
from pennylane import numpy as np
import pennylane as qml

## Digitized-counterdiabatic quantum computing

Following the paper : https://journals.aps.org/prresearch/abstract/10.1103/PhysRevResearch.4.013141 

Class for the graphs. 

In [ ]:
class Graph:
    
    def __init__(self, n, g_type, seed = 1906):
        self.n = n
        self.edges = {}
        self.g_type = g_type
        self.seed = seed 
        self.generate_graph()


    def generate_graph(self):

        if self.g_type == "3REGULAR":
            g_nx = nx.random_regular_graph(3, self.n, self.seed)
            for k in g_nx.edges:
                self.edges[(k[0], k[1])] = 1
        else:
            for i in range(self.n - 1):
                for j in range(i + 1, self.n):
                    self.edges[(i, j)] = (
                        float(rd.choice([-1, 1]))
                        if self.g_type == "SK"
                        else float(rd.uniform(-1, 1))
                    )

    def get_ground_energy(self):
        h = {i: 0 for i in range(self.n)}
        j = {k: -v for k, v in self.edges.items()}
        sampleset = dimod.ExactSolver().sample_ising(h, self.edges)
        return sampleset
        



We define the different counter-diabatic terms. 

In [ ]:
def y_cd_term(params, g):
    for i in range(g.n):
        qml.RY(params, wires=i)

def yz_cd_term(params, g, inverse=False):
    for e in g.edges.keys():
        qml.RX(np.pi/2,wires=e[0] if not inverse else e[1])
        qml.CNOT(wires=[e[0],e[1]])
        qml.RZ(-g.edges[e]*params,wires=e[1])
        qml.CNOT(wires=[e[0],e[1]])
        qml.RX(-np.pi/2,wires=e[0] if not inverse else e[1])

def xy_cd_term(params, g, inverse=False):
    for e in g.edges.keys():
        qml.Hadamard(wires=e[0])
        qml.RX(np.pi/2,wires=e[1] if not inverse else e[0])
        qml.CNOT(wires=[e[0],e[1]])
        qml.RZ(-g.edges[e]*params,wires=e[1])
        qml.CNOT(wires=[e[0],e[1]])
        qml.RX(-np.pi/2,wires=e[1] if not inverse else e[0])
        qml.Hadamard(wires=e[0])

cd_terms_dict = {
    'Y': y_cd_term,
    'YZ': yz_cd_term,
    'ZY':lambda params, g: yz_cd_term(params, g, inverse=True),
    'XY': xy_cd_term,
    'YX' : lambda params, g: xy_cd_term(params, g, inverse=True)
}

Function definition for the circuit. 

In [ ]:
def dc_qaoa(g, n_layers, cd_term=None):

    dev = qml.device("default.qubit", wires=g.n)
    @qml.qnode(dev)
    def circuit(params):
        for i in range(g.n):
            qml.Hadamard(wires=i)

        for j in range(n_layers):
            for e in g.edges.keys():
                qml.CNOT(wires=[e[0], e[1]])
                qml.RZ(-g.edges[e] *params[0][j], wires = e[1])
                qml.CNOT(wires=[e[0], e[1]])

            for k in range(g.n):
                qml.RX(params[1][j], wires=k)

            if cd_term is not None:
                cd_terms_dict[cd_term](params[2][j], g)

        H = qml.Hamiltonian([-g.edges[e] for e in g.edges.keys()],
                    [qml.PauliZ(e[0]) @ qml.PauliZ(e[1]) for e in g.edges.keys()])
        return qml.expval(H)
    
    return circuit



We create a function for optimisation. 

In [ ]:
def minimise(n_graphs, n_nodes, graph_type, seed, n_layers, cd_term, learning_rate, steps):
    g_list = [Graph(n=n_nodes, g_type=graph_type, seed=seed*(i + 1004)) for i in range(n_graphs)]
    def cost(params):
        total = 0
        for g in g_list:
            circuit = dc_qaoa(g, n_layers, cd_term=cd_term)  # choose your cd_term
            total += circuit(params)
        return total / len(g_list)  # mean cost over all graphs
    
    opt = qml.AdagradOptimizer(stepsize=learning_rate)
    params = np.random.rand(2 if cd_term is None else 3, n_layers, requires_grad=True)
    for it in range(steps):
        params = opt.step(cost, params)  # update parameters
        current_cost = cost(params)
        print(f"Iteration {it+1}: cost = {current_cost:.6f}")
    



We make a trial: 

In [ ]:
minimise(20, 4, 'weighted_uniform', 9000, 2, None, 0.1, 100)